# 📄 Agentic AI for Disaster Scene Analysis (G1-11) — Kaggle Master Execution

**Team Members:** Momin (Group Head), Abdurrehman, Zuhaak  
**Hardware Target:** Kaggle GPU (T4 / P100)  
**Execution Protocol:** RUNBOOK Stages 0 through 10

In [ ]:
# 1. Clone or update the repository from GitHub
import os
from pathlib import Path

repo_path = Path("/kaggle/working/A14015-Project-G1-11")
if not repo_path.exists():
    !git clone https://github.com/IbneRaees/A14015-Project-G1-11.git /kaggle/working/A14015-Project-G1-11
else:
    !cd /kaggle/working/A14015-Project-G1-11 && git pull origin main

%cd /kaggle/working/A14015-Project-G1-11

# 2. Install required dependencies
!pip install -q -r requirements.txt

# 3. Verify PyTorch GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")


In [ ]:
import os
from pathlib import Path

# 1. Automatically locate FloodNet dataset inside /kaggle/input
print("[+] Searching for FloodNet dataset in /kaggle/input/...")
possible_org = list(Path("/kaggle/input").rglob("train-org-img"))

if possible_org:
    KAGGLE_PATH = str(possible_org[0].parent.parent)
    print(f"[+] Found FloodNet dataset at: {KAGGLE_PATH}")
else:
    # Check fallback folder names
    possible_train = list(Path("/kaggle/input").rglob("train"))
    if possible_train:
        KAGGLE_PATH = str(possible_train[0].parent)
        print(f"[+] Found dataset root at: {KAGGLE_PATH}")
    else:
        raise FileNotFoundError(
            "\n\n[!] ERROR: FloodNet dataset not found in /kaggle/input!\n"
            "Do NOT download inside the notebook (it exceeds the 20 GB disk limit).\n"
            "To attach it for FREE (0 GB used):\n"
            "  1. Click '+ Add Input' on the right sidebar of this Kaggle notebook.\n"
            "  2. Search for: 'FloodNet Challenge' or 'aleatbm/aerial-imagery-dataset-floodnet-challenge'\n"
            "  3. Click '+' to add it to your session.\n"
            "  4. Re-run this cell!\n"
        )

os.environ["DATASET_ROOT"] = KAGGLE_PATH
print(f"[+] Using DATASET_ROOT: {KAGGLE_PATH}")

# 2. Run Stage 0 preflight check & split verification (Expects 1445 / 450 / 448)
!python -m data.floodnet --root "$DATASET_ROOT" --build-manifest
!python -m data.floodnet --root "$DATASET_ROOT" --verify

# 3. Run Stage 0.5 Supervised YOLO preparation (creates symlinks in datasets/floodnet_yolo)
!python -m data.prepare_yolo --root "$DATASET_ROOT" --out datasets/floodnet_yolo


In [ ]:
# Stage 1: Fine-tune YOLOv8x-Seg across seeds 42, 123, 456
!python -m experiments.train_baseline --seed 42 --data-root "$DATASET_ROOT"
!python -m experiments.train_baseline --seed 123 --data-root "$DATASET_ROOT"
!python -m experiments.train_baseline --seed 456 --data-root "$DATASET_ROOT"

# Stage 2: Evaluate E1 baseline and export predicted label maps
!python -m experiments.run_baseline --split val --data-root "$DATASET_ROOT" --save-preds results/preds/e1_val
!python -m experiments.run_baseline --data-root "$DATASET_ROOT" --save-preds results/preds/e1


In [ ]:
# Stage 3: Train Evidential and Softmax uncertainty heads across all 3 seeds
!python -m experiments.train_uncertainty_head --head evidential --seed 42 --data-root "$DATASET_ROOT"
!python -m experiments.train_uncertainty_head --head softmax --seed 42 --data-root "$DATASET_ROOT"

!python -m experiments.train_uncertainty_head --head evidential --seed 123 --data-root "$DATASET_ROOT"
!python -m experiments.train_uncertainty_head --head softmax --seed 123 --data-root "$DATASET_ROOT"

!python -m experiments.train_uncertainty_head --head evidential --seed 456 --data-root "$DATASET_ROOT"
!python -m experiments.train_uncertainty_head --head softmax --seed 456 --data-root "$DATASET_ROOT"

# Stage 4: Run Gap 2 calibration evaluation and plot reliability curves
!python -m experiments.run_gap2_calibration --data-root "$DATASET_ROOT" --plot


In [ ]:
import os

# Stage 5: Sweep min_blocked_run_px on validation predictions
os.makedirs("results/tune", exist_ok=True)
for m in [10, 15, 20, 30, 40]:
    print(f"\n=== Running Gap 1 sweep for min_blocked_run_px={m} ===")
    !python -m experiments.run_gap1_routes --data-root "$DATASET_ROOT" --split val \
        --preds e1_s42=results/preds/e1_val/seed42 --min-blocked-run-px {m} \
        --output results/tune/routes_val_m{m}.json --queries-out results/tune/queries_val.json

!python -m experiments.summarize_sweep results/tune/routes_val_m*.json --system e1_s42/graph


In [ ]:
# Stage 6: Run E2 zero-shot monolithic VLM on test split (Resumable via logs/e2/tiles.jsonl)
!python -m experiments.run_vlm_zeroshot --data-root "$DATASET_ROOT"


In [ ]:
# Stage 7: Run full 3-agent E3 pipeline across 3 seeds
!python -m experiments.run_e3 --data-root "$DATASET_ROOT" --save-preds results/preds/e3

# Stage 8: Run E5 component ablations
!python -m experiments.run_e3 --data-root "$DATASET_ROOT" --ablation no_gap1
!python -m experiments.run_e3 --data-root "$DATASET_ROOT" --ablation no_gap2
!python -m experiments.run_e3 --data-root "$DATASET_ROOT" --ablation no_gap3


In [ ]:
# Stage 9: Run route-query benchmark on test set (Set --min-blocked-run-px to your Stage 5 optimal value)
!python -m experiments.run_gap1_routes --data-root "$DATASET_ROOT" --min-blocked-run-px 20 \
    --preds e1_s42=results/preds/e1/seed42 --preds e1_s123=results/preds/e1/seed123 --preds e1_s456=results/preds/e1/seed456 \
    --preds e3_s42=results/preds/e3/seed42 --preds e3_s123=results/preds/e3/seed123 --preds e3_s456=results/preds/e3/seed456

!python -m experiments.run_gap1_routes --data-root "$DATASET_ROOT" --min-blocked-run-px 20 \
    --preds e1_s42=results/preds/e1/seed42 --preds e1_s123=results/preds/e1/seed123 --preds e1_s456=results/preds/e1/seed456 \
    --preds e3_s42=results/preds/e3/seed42 --preds e3_s123=results/preds/e3/seed123 --preds e3_s456=results/preds/e3/seed456 --vlm


In [ ]:
# Stage 10: Compile statistical comparison tables (Markdown & LaTeX)
!python -m experiments.compare_results

# Create a zip archive of all results, figures, and tables for easy download
import shutil
!zip -r /kaggle/working/G1-11_Final_Results.zip results/ logs/ || python -c "import shutil; shutil.make_archive('/kaggle/working/G1-11_Final_Results', 'zip', '/kaggle/working/A14015-Project-G1-11')"
print("Final results package created at /kaggle/working/G1-11_Final_Results.zip")
